# 02 - Fase 3 (preparacion): entrenar Chemprop (DG de solvatacion)

Modelo de **2 componentes**: `(solvente, soluto) -> DG_solv [kcal/mol]`. Se entrena una vez; el nb 03 lo usa en cada ronda.

**Datos (los subes como Kaggle Dataset):** CSV con columna SMILES de solvente, de soluto y DG en kcal/mol. Fuentes: **CombiSolv-Exp** (Vermeire & Green 2021, ~10k pares), **MNSol** (academico, previa solicitud), o datos propios de ILs/DES (los mas utiles). Casi ningun dataset publico tiene iones metal-oxo ni muchos ILs/DES; la celda de cobertura lo cuantifica.

**Settings:** GPU · Internet On · Add Input -> tu dataset. Save & Run All (Commit).

In [ ]:
# ==== Setup comun: codigo en /tmp (efimero), estado en /kaggle/working/alumina (se guarda) ====
import os, glob, json, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/santiagovelcasallas/Data_Science_example_01.git"
BRANCH   = "claude/affectionate-johnson-q9wtj0"   # cambia a "main" tras el merge

ROOT = Path("/kaggle/working/alumina")
CODE = Path("/tmp/repo")
PKG  = CODE / "alumina_pipeline"
for d in ["data", "models", "priors", "runs"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)
os.environ["ALUMINA_ROOT"] = str(ROOT)

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

token = secret("GITHUB_TOKEN")   # solo si el repo es privado
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
if not CODE.exists():
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, str(CODE)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git clone fallo: " + r.stderr[-300:])
print("Codigo en", PKG)

In [ ]:
# ==== Instalacion (~3-6 min). Se fija la version de torch de Kaggle para no romper CUDA ====
!pip freeze | grep -E "^torch==" > /tmp/constraints.txt; cat /tmp/constraints.txt
!pip install -q -c /tmp/constraints.txt -r {PKG}/kaggle/requirements-kaggle.txt
!pip install -q --no-deps -e {PKG}
!python -c "import torch,rdkit,ase,botorch,alumina_pipeline as a; print('torch',torch.__version__,'| GPU:',torch.cuda.is_available())"

In [ ]:
CSV_PATTERN = "*.csv"
SOLVENT_COL, SOLUTE_COL, TARGET_COL = "solvent", "solute", "dG"   # ajusta a tu CSV
TARGET_UNITS = "kcal/mol"   # "kJ/mol" -> se convierte
EPOCHS, ENSEMBLE = 100, 3

csvs = sorted(glob.glob("/kaggle/input/**/" + CSV_PATTERN + "", recursive=True))
print("\n".join(csvs) or "[!] Sin CSV: Add Input -> tu dataset")

In [ ]:
import pandas as pd
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

df = pd.read_csv(csvs[0]).rename(columns={SOLVENT_COL:"solvent", SOLUTE_COL:"solute", TARGET_COL:"dG"})
df = df[["solvent","solute","dG"]].dropna()
if TARGET_UNITS == "kJ/mol": df["dG"] = df["dG"] / 4.184

def canon(s):
    m = Chem.MolFromSmiles(str(s));  return Chem.MolToSmiles(m) if m else None
n0 = len(df)
df["solvent"], df["solute"] = df["solvent"].map(canon), df["solute"].map(canon)
df = df.dropna()
print(f"Filas validas: {len(df)}/{n0} | solventes unicos: {df.solvent.nunique()} | solutos unicos: {df.solute.nunique()}")

ionic = df.solvent.map(lambda s: "." in s or "+" in s).mean()
print(f"Cobertura -> solventes ionicos/mezclas: {ionic:.1%}")
if ionic < 0.01: print("[!] Casi no hay solventes ionicos: ILs/DES seran extrapolaciones.")
train_csv = ROOT / "data/solvation_train.csv"; df.to_csv(train_csv, index=False)

In [ ]:
!chemprop train --data-path {train_csv} --smiles-columns solvent solute --target-columns dG \
  --task-type regression --epochs {EPOCHS} --ensemble-size {ENSEMBLE} \
  --output-dir {ROOT}/models/chemprop_solvation 2>&1 | grep -viE "warning|it/s" | tail -n 40

In [ ]:
# Chequeo de cordura: agua en agua ~ -6.3 kcal/mol
test = pd.DataFrame({"solvent":["O","CO","O"], "solute":["O","O","[OH-]"],
                     "nota":["agua/agua ~-6.3","agua/metanol","OH-/agua"]})
test[["solvent","solute"]].to_csv("/tmp/test.csv", index=False)
!chemprop predict --test-path /tmp/test.csv --preds-path /tmp/tp.csv --model-paths {ROOT}/models/chemprop_solvation --smiles-columns solvent solute 2>&1 | tail -n 1
tp = pd.read_csv("/tmp/tp.csv")
test["pred_kcal_mol"] = tp[[c for c in tp.columns if c not in ("solvent","solute")][0]].values
test

## Que guardar

| Archivo | Para que | Imprescindible |
|---|---|---|
| `alumina/models/chemprop_solvation/` (carpeta) | El modelo que usa el nb 03 | **Si** |
| RMSE/MAE de test (impresos arriba) | Cuanto confiar en DG | **Si** (anotalos) |
| `alumina/data/solvation_train.csv` | Reproducibilidad | Recomendado |

En el nb 03: Add Input -> Notebook Output -> este notebook.